In [14]:
from models.TransBTS.TransBTS_downsample8x_skipconnection import TransBTS

In [15]:
_, model = TransBTS(_conv_repr=True, _pe_type="learned")

In [16]:
model.eval()

BTS(
  (linear_encoding): Linear(in_features=128, out_features=512, bias=True)
  (position_encoding): LearnedPositionalEncoding()
  (pe_dropout): Dropout(p=0.1, inplace=False)
  (transformer): TransformerModel(
    (net): IntermediateSequential(
      (0): Residual(
        (fn): PreNormDrop(
          (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (fn): SelfAttention(
            (qkv): Linear(in_features=512, out_features=1536, bias=False)
            (attn_drop): Dropout(p=0.1, inplace=False)
            (proj): Linear(in_features=512, out_features=512, bias=True)
            (proj_drop): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (1): Residual(
        (fn): PreNorm(
          (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (fn): FeedForward(
            (net): Sequential(
              (0): Linear(in_features=512, out_features=4096, bias=True)
              (1

In [50]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import nibabel as nib
import pandas as pd
class TransBTS_model_dataset_enoder_only(Dataset):
    def __init__(self, BrainMRIFilesPath, CSVPath, transform=None):        
        self.BrainMRIFilesPath = BrainMRIFilesPath
        self.CSVPath = CSVPath
        self.transform = transform
        self.df = pd.read_excel(self.CSVPath , engine='openpyxl')
        
        self.Files = os.listdir(self.BrainMRIFilesPath)

        self.ID_To_Filename = {}
        self.IDs = []
        self.Lables = {}
        for file in self.Files:
            strc = file.split('-')
            self.ID_To_Filename[int(strc[0][-3:])] = file
            self.IDs.append(int(strc[0][-3:]))
            self.Lables[int(strc[0][-3:])] = np.random.randint(0, 2, size=(4096, 512), dtype=np.int32)

    def __len__(self):
        return len(self.IDs)

    def __getitem__(self, index):
        ID = self.IDs[index]
        filename = self.ID_To_Filename[ID]
        file_path = os.path.join(self.BrainMRIFilesPath ,filename)
        if self.transform is not None:
            image = transforms(file_path)              # returns torch.Tensor
            array = image.numpy()[0]                    
        
        age = self.df[self.df['IXI_ID'] == ID]['AGE'].values[0]
        Label = self.Lables[ID]
        return array, Label
        

In [51]:
from monai.transforms import LoadImage, Compose, Spacing, Resize, EnsureChannelFirst, ScaleIntensity
import numpy as np
import torch
transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),                   # shape: (1, H, W, D)
    ScaleIntensity(),                       # optional: normalize to 0-1
    Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear"),  # resample to 1mm spacing
    Resize(spatial_size=(128, 128, 128), mode='trilinear')  # resize to 128³
])

In [52]:
BrainMRIFilesPath = r"C:\Users\Rishabh\Documents\PreprocessedData1mm"
CSVPath = os.path.join(r'C:\Users\Rishabh\Documents\TransBTS', 'IXI.xlsx')
ds = TransBTS_model_dataset_enoder_only(BrainMRIFilesPath, CSVPath , transforms)

In [53]:
ImageVolume, age = ds[0]

In [54]:
ImageVolume.shape

(128, 128, 128)

In [56]:
age.shape

(4096, 512)

In [57]:
ImageVolume = torch.from_numpy(ImageVolume)
ImageVolume= ImageVolume.unsqueeze(0).unsqueeze(0)

In [62]:
import torch.nn as nn

criterion = nn.BCELoss()
# loss = criterion(predicted, labels)


In [58]:
with torch.no_grad():
    encoder_output, intmd_encoder_outputs = model(ImageVolume)  

In [63]:
age = torch.from_numpy(age)

In [64]:
age.shape

torch.Size([4096, 512])

In [60]:
encoder_output = torch.sigmoid(encoder_output)

In [66]:
encoder_output = encoder_output.squeeze(0)

In [67]:
encoder_output.shape

torch.Size([4096, 512])

In [69]:
loss = criterion(encoder_output, age.float())

In [70]:
loss

tensor(0.8064)

In [59]:
torch.unique(encoder_output)

tensor([-3.9419, -3.9299, -3.9129,  ...,  3.8960,  3.9427,  4.0116])

In [37]:
encoder_output.shape

torch.Size([1, 4096, 512])

In [38]:
import torch

# Generate tensor of shape (4096, 512) with random 0s and 1s
tensor = torch.randint(0, 2, (4096, 512), dtype=torch.int)

print(tensor)
print(tensor.shape)  # Should print: torch.Size([4096, 512])


tensor([[0, 0, 0,  ..., 0, 1, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [1, 0, 0,  ..., 1, 0, 1],
        ...,
        [0, 1, 0,  ..., 0, 0, 0],
        [1, 1, 0,  ..., 1, 0, 0],
        [1, 0, 0,  ..., 0, 0, 1]], dtype=torch.int32)
torch.Size([4096, 512])
